# 02 — Data Quality and Temporal Split

## 1. Objective

Assess the quality, consistency, and temporal integrity of the SWaT dataset before exploratory analysis and anomaly-detection modelling.

The assessment covers data provenance, schema consistency, missing and invalid values, duplicate records, timestamp ordering, label integrity, and reconciliation between the separate and merged CSV files.

The findings will guide data preparation and the design of chronological evaluation partitions. Any data-quality issue that could affect model development or evaluation will be documented before proceeding.

## 2. Scope

This notebook will:

- Establish the dataset files and their structural characteristics.
- Verify column names, schema consistency, and data types.
- Assess missing values, invalid values, and duplicate records.
- Investigate timestamp ordering and temporal continuity.
- Examine the distribution and consistency of anomaly labels.
- Reconcile `normal.csv`, `attack.csv`, and `merged.csv`.
- Define an appropriate chronological partitioning strategy.
- Document data-quality limitations and their implications for subsequent analysis.

## 3. Data Handling Principles

- Preserve the original files in `data/raw/` without modification.
- Use the existing project paths rather than hardcoded machine-specific paths.
- Process large CSV files in chunks when a complete in-memory load is unnecessary.
- Distinguish verified findings from unresolved questions.
- Preserve temporal order when designing evaluation partitions.
- Document any preprocessing decisions that could affect anomaly detection or evaluation.

## 4. Expected Outcome

By the end of this notebook, the dataset's principal quality issues and temporal characteristics should be documented, file consistency should be assessed, and the rationale for the evaluation partitions should be established.

The notebook will provide the data-quality evidence needed before proceeding to **Notebook 03 — Exploratory Analysis**.

In [10]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate data/raw from the current directory: {CURRENT_DIR}"
    )

DATA_DIR = PROJECT_ROOT / "data" / "raw"

FILES = {
    "normal": DATA_DIR / "normal.csv",
    "attack": DATA_DIR / "attack.csv",
    "merged": DATA_DIR / "merged.csv",
}

missing_files = [
    path for path in FILES.values()
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Required dataset files not found:\n"
        + "\n".join(str(path) for path in missing_files)
    )

# print(f"Project root: {PROJECT_ROOT}")
# print(f"Raw data directory: {DATA_DIR}")

for name, path in FILES.items():
    print(f"{name}: {path.name} ({path.stat().st_size:,} bytes)")

normal: normal.csv (402,374,633 bytes)
attack: attack.csv (15,162,438 bytes)
merged: merged.csv (426,855,390 bytes)


## 5. Dataset Inventory

The raw dataset directory contains three CSV files: `normal.csv`, `attack.csv`, and `merged.csv`.

| File | Size (bytes) | Intended role |
|---|---:|---|
| `normal.csv` | 402,374,633 | Records designated as normal operation |
| `attack.csv` | 15,162,438 | Records associated with attack scenarios |
| `merged.csv` | 426,855,390 | Combined dataset containing normal and attack records |

The files are stored in `data/raw/`. Their existence and file sizes have been verified programmatically.

These checks establish the initial file inventory only. They do not establish that the files contain valid records, that `merged.csv` is an exact combination of the other two files, or that their timestamps are correctly ordered. Those properties require separate validation.

In [11]:
import csv

schemas = {}

for name, path in FILES.items():
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)
        headers = next(reader)

    normalized_headers = [header.strip() for header in headers]

    schemas[name] = normalized_headers

    print(f"{name}: {len(headers)} columns")

    if len(normalized_headers) != len(set(normalized_headers)):
        raise ValueError(f"{name} contains duplicate column names after trimming whitespace.")

reference_schema = schemas["normal"]

for name, schema in schemas.items():
    if schema != reference_schema:
        raise ValueError(f"{name} has a different column order or schema from normal.csv.")

print("\nPASS: All three files have matching column names and column order.")
print("PASS: No duplicate column names were found after trimming whitespace.")

normal: 53 columns
attack: 53 columns
merged: 53 columns

PASS: All three files have matching column names and column order.
PASS: No duplicate column names were found after trimming whitespace.


## 6. Schema Consistency

All three raw CSV files contain **53 columns**. After removing leading and trailing whitespace from the column names, the files have identical column names and column order.

No duplicate column names were identified after normalization.

The raw headers contain leading whitespace, so column names should be normalized during analysis rather than modifying the original CSV files. The raw files remain preserved unchanged.

Schema consistency is therefore **PASS** across `normal.csv`, `attack.csv`, and `merged.csv`.

In [12]:
import pandas as pd

CHUNK_SIZE = 100_000

missing_results = {}

for name, path in FILES.items():
    missing_counts = pd.Series(dtype="int64")
    row_count = 0

    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE):
        chunk.columns = chunk.columns.str.strip()
        row_count += len(chunk)

        chunk_missing = chunk.isna().sum()

        if missing_counts.empty:
            missing_counts = chunk_missing.astype("int64")
        else:
            missing_counts = missing_counts.add(chunk_missing, fill_value=0).astype("int64")

    missing_results[name] = {
        "rows": row_count,
        "missing_total": int(missing_counts.sum()),
        "columns_with_missing": missing_counts[missing_counts > 0].sort_values(ascending=False),
    }

    print(f"{name}:")
    print(f"  rows: {row_count:,}")
    print(f"  total missing values: {int(missing_counts.sum()):,}")
    print(f"  columns with missing values: {len(missing_counts[missing_counts > 0])}")

    if (missing_counts > 0).any():
        print(missing_counts[missing_counts > 0])
    else:
        print("  No missing values found.")

    print()

normal:
  rows: 1,387,098
  total missing values: 6,942,600
  columns with missing values: 7
MV101     991800
AIT201    991800
MV201     991800
P201      991800
P202      991800
P204      991800
MV303     991800
dtype: int64

attack:
  rows: 54,621
  total missing values: 0
  columns with missing values: 0
  No missing values found.

merged:
  rows: 1,441,719
  total missing values: 6,942,600
  columns with missing values: 7
MV101     991800
AIT201    991800
MV201     991800
P201      991800
P202      991800
P204      991800
MV303     991800
dtype: int64



## 7. Missing Value Assessment

The missing-value assessment identified missing observations in seven variables:

- `MV101`
- `AIT201`
- `MV201`
- `P201`
- `P202`
- `P204`
- `MV303`

Each affected variable contains **991,800 missing values** in `normal.csv`, giving **6,942,600 missing cells** across the seven variables.

`attack.csv` contains **no missing values**. The same 6,942,600 missing cells are present in `merged.csv`, which is consistent with the missing values originating from the normal-operation records.

These missing values are not removed or imputed at this stage. Their temporal and process context must be investigated before determining whether they represent genuine sensor/actuator availability, structural missingness, or a data-quality problem.

**Status: Investigation required.**

In [17]:
MISSING_COLUMNS = [
    "MV101", "AIT201", "MV201", "P201",
    "P202", "P204", "MV303",
]

with FILES["normal"].open("r", encoding="utf-8-sig", newline="") as file:
    raw_headers = next(csv.reader(file))

selected_columns = [
    header for header in raw_headers
    if header.strip() == "Timestamp"
    or header.strip() in MISSING_COLUMNS
]

if len(selected_columns) != len(MISSING_COLUMNS) + 1:
    raise ValueError(
        f"Expected {len(MISSING_COLUMNS) + 1} columns, "
        f"but found {len(selected_columns)}."
    )

missing_time_results = []
total_rows_with_missing_values = 0
total_invalid_timestamps = 0

for chunk in pd.read_csv(
    FILES["normal"],
    usecols=selected_columns,
    chunksize=CHUNK_SIZE,
    encoding="utf-8-sig",
):
    chunk.columns = chunk.columns.str.strip()

    chunk["Timestamp"] = pd.to_datetime(
        chunk["Timestamp"].str.strip(),
        format="%d/%m/%Y %I:%M:%S %p",
        errors="coerce",
    )

    total_invalid_timestamps += int(chunk["Timestamp"].isna().sum())

    missing_mask = chunk[MISSING_COLUMNS].isna().any(axis=1)
    affected = chunk.loc[missing_mask, "Timestamp"]

    total_rows_with_missing_values += int(missing_mask.sum())

    if missing_mask.any():
        missing_time_results.append({
            "first_timestamp": affected.min(),
            "last_timestamp": affected.max(),
            "affected_rows": int(missing_mask.sum()),
        })

print(f"Total rows with missing values: {total_rows_with_missing_values:,}")
print(f"Total invalid timestamps: {total_invalid_timestamps:,}")
print(f"Chunks containing missing values: {len(missing_time_results)}")

if missing_time_results:
    print(
        "Overall first timestamp:",
        min(r["first_timestamp"] for r in missing_time_results),
    )
    print(
        "Overall last timestamp:",
        max(r["last_timestamp"] for r in missing_time_results),
    )

if total_rows_with_missing_values != 991_800:
    raise ValueError(
        "The observed number of rows with missing values does not match "
        "the previously recorded count of 991,800."
    )

if total_invalid_timestamps:
    raise ValueError(
        f"Found {total_invalid_timestamps:,} invalid timestamps; "
        "investigate before interpreting the time range."
    )

Total rows with missing values: 991,800
Total invalid timestamps: 0
Chunks containing missing values: 11
Overall first timestamp: 2015-12-22 16:00:00
Overall last timestamp: 2015-12-28 09:59:59


## 8. Temporal Assessment of Missing Values

The temporal investigation examined the seven variables previously identified as containing missing values in `normal.csv`.

The check identified 991,800 rows containing at least one missing value in these variables. This matches the count established during the initial missing-value assessment. No invalid timestamps were detected.

The affected rows span the following observed timestamp range:

- **First affected timestamp:** 22 December 2015, 16:00:00
- **Last affected timestamp:** 28 December 2015, 09:59:59

The affected rows were found in 11 processing chunks. These findings establish the overall time range of the missingness but do not, by themselves, establish that every affected row belongs to one continuous block.

The last affected timestamp is `2015-12-28 09:59:59`, one second before `2015-12-28 10:00:00` — the timestamp recorded in the reference notebook as the first row of `normal.csv`. This is consistent with the missing block occupying the beginning of the file but requires direct confirmation.

No missing values have been imputed, and no rows have been removed. The cause and operational significance of the missingness remain unresolved and must be considered before selecting a treatment strategy.

**Status: Temporal range established; cause not yet determined.**

## 9. Timestamp Ordering Assessment

A chunked scan of `normal.csv` confirmed two backward timestamp transitions and established the exact extent of the missing-value block.

| Previous data row | Next data row | Previous timestamp | Next timestamp |
|---:|---:|---|---|
| 395,298 | 395,299 | `2016-01-02 14:59:59` | `2015-12-22 16:00:00` |
| 892,098 | 892,099 | `2015-12-28 09:59:59` | `2015-12-22 16:30:00` |

Data-row numbers are 1-indexed and exclude the header.

### Relationship to the Missing-Value Pattern

The scan identified one continuous block of **991,800 rows** containing missing values across the seven previously identified variables. The block begins at data row 395,299 and extends to the end of `normal.csv`, at data row 1,387,098.

The block's first and last timestamps are `2015-12-22 16:00:00` and `2015-12-28 09:59:59`, respectively. Both backward timestamp transitions occur within this block's boundaries: the first marks its beginning, while the second marks a restart at `2015-12-22 16:30:00`.

These findings confirm a positional relationship between the missing values and the timestamp discontinuities. They do not, by themselves, establish the cause of the missingness or explain the overlapping timestamp sequences.

### Implications for Data Preparation

The original row order must be preserved while the file structure is investigated. Sorting solely by timestamp or removing repeated timestamps could alter the relationship between observations and their original positions.

The next validation step is to examine `merged.csv` and reconcile its row structure with `normal.csv` and `attack.csv`. No rows will be removed, reordered, or imputed until the structure and implications of these findings have been established.

**Status: Missing-value block and timestamp discontinuities verified in `normal.csv`; cause and cross-file reconciliation remain under investigation.**

In [19]:
with FILES["normal"].open("r", encoding="utf-8-sig", newline="") as file:
    raw_headers = next(csv.reader(file))

selected_columns = [
    header for header in raw_headers
    if header.strip() == "Timestamp"
    or header.strip() in MISSING_COLUMNS
]

if len(selected_columns) != 8:
    raise ValueError(f"Expected 8 columns; found {len(selected_columns)}.")

missing_runs = []
backward_transitions = []
total_missing_rows = 0
total_invalid_timestamps = 0
total_rows = 0
previous_timestamp = None
previous_missing = False

for chunk in pd.read_csv(
    FILES["normal"],
    usecols=selected_columns,
    chunksize=CHUNK_SIZE,
    encoding="utf-8-sig",
):
    chunk.columns = chunk.columns.str.strip()

    timestamps = pd.to_datetime(
        chunk["Timestamp"].str.strip(),
        format="%d/%m/%Y %I:%M:%S %p",
        errors="coerce",
    )

    total_invalid_timestamps += int(timestamps.isna().sum())

    missing_mask = chunk[MISSING_COLUMNS].isna().any(axis=1).to_numpy()
    total_missing_rows += int(missing_mask.sum())

    row_start = total_rows + 1

    if previous_timestamp is not None and len(timestamps):
        if timestamps.iloc[0] < previous_timestamp:
            backward_transitions.append({
                "previous_data_row": row_start - 1,
                "next_data_row": row_start,
                "previous_timestamp": previous_timestamp,
                "next_timestamp": timestamps.iloc[0],
            })

    backward_positions = np.flatnonzero(
        timestamps.diff().lt(pd.Timedelta(0)).to_numpy()
    )

    for position in backward_positions:
        backward_transitions.append({
            "previous_data_row": row_start + int(position) - 1,
            "next_data_row": row_start + int(position),
            "previous_timestamp": timestamps.iloc[position - 1],
            "next_timestamp": timestamps.iloc[position],
        })

    boundaries = np.flatnonzero(
        np.diff(np.r_[False, missing_mask, False].astype(np.int8))
    )

    for start, end in boundaries.reshape(-1, 2):
        run_end = row_start + int(end) - 1

        if start == 0 and previous_missing:
            missing_runs[-1]["last_data_row"] = run_end
            missing_runs[-1]["last_timestamp"] = timestamps.iloc[end - 1]
            missing_runs[-1]["rows"] += int(end - start)
        else:
            missing_runs.append({
                "first_data_row": row_start + int(start),
                "last_data_row": run_end,
                "first_timestamp": timestamps.iloc[start],
                "last_timestamp": timestamps.iloc[end - 1],
                "rows": int(end - start),
            })

    if len(missing_mask):
        previous_missing = bool(missing_mask[-1])

    if len(timestamps):
        previous_timestamp = timestamps.iloc[-1]

    total_rows += len(chunk)

print(f"Total data rows: {total_rows:,}")
print(f"Rows containing missing values: {total_missing_rows:,}")
print(f"Invalid timestamps: {total_invalid_timestamps:,}")

if total_missing_rows != 991_800:
    raise ValueError(
        "Missing-row count does not match the previously verified count of 991,800."
    )

if total_invalid_timestamps:
    raise ValueError(
        f"Found {total_invalid_timestamps:,} invalid timestamps across the file."
    )

print("\nContiguous runs of rows containing missing values:")
for run in missing_runs:
    print(run)

print("\nBackward timestamp transitions (1-indexed data rows, excluding header):")
for transition in backward_transitions:
    print(transition)

Total data rows: 1,387,098
Rows containing missing values: 991,800
Invalid timestamps: 0

Contiguous runs of rows containing missing values:
{'first_data_row': 395299, 'last_data_row': 1387098, 'first_timestamp': Timestamp('2015-12-22 16:00:00'), 'last_timestamp': Timestamp('2015-12-28 09:59:59'), 'rows': 991800}

Backward timestamp transitions (1-indexed data rows, excluding header):
{'previous_data_row': 395298, 'next_data_row': 395299, 'previous_timestamp': Timestamp('2016-01-02 14:59:59'), 'next_timestamp': Timestamp('2015-12-22 16:00:00')}
{'previous_data_row': 892098, 'next_data_row': 892099, 'previous_timestamp': Timestamp('2015-12-28 09:59:59'), 'next_timestamp': Timestamp('2015-12-22 16:30:00')}


In [21]:
SEGMENT_A_START = 395_299
SEGMENT_A_END = 892_098
SEGMENT_B_START = 892_099
SEGMENT_B_END = 1_387_098

with FILES["normal"].open("r", encoding="utf-8-sig", newline="") as file:
    raw_headers = next(csv.reader(file))

columns_to_read = [
    header
    for header in raw_headers
    if header.strip() == "Timestamp"
    or header.strip() not in MISSING_COLUMNS
]

segment_a_parts = []
segment_b_parts = []
row_offset = 0

for chunk in pd.read_csv(
    FILES["normal"],
    usecols=columns_to_read,
    chunksize=CHUNK_SIZE,
    encoding="utf-8-sig",
):
    chunk.columns = chunk.columns.str.strip()

    chunk_start = row_offset + 1
    chunk_end = row_offset + len(chunk)

    for start, end, parts in (
        (SEGMENT_A_START, SEGMENT_A_END, segment_a_parts),
        (SEGMENT_B_START, SEGMENT_B_END, segment_b_parts),
    ):
        overlap_start = max(start, chunk_start)
        overlap_end = min(end, chunk_end)

        if overlap_start <= overlap_end:
            parts.append(
                chunk.iloc[
                    overlap_start - chunk_start:
                    overlap_end - chunk_start + 1
                ].copy()
            )

    row_offset += len(chunk)

if row_offset != 1_387_098:
    raise ValueError(f"Expected 1,387,098 data rows; found {row_offset:,}.")

segment_a_df = pd.concat(segment_a_parts, ignore_index=True)
segment_b_df = pd.concat(segment_b_parts, ignore_index=True)

if len(segment_a_df) != 496_800 or len(segment_b_df) != 495_000:
    raise ValueError("Extracted segment lengths do not match the verified boundaries.")

OFFSET = len(segment_a_df) - len(segment_b_df)

if OFFSET != 1_800:
    raise ValueError(f"Expected an offset of 1,800 rows; found {OFFSET:,}.")

aligned_a = segment_a_df.iloc[OFFSET:].reset_index(drop=True)
aligned_b = segment_b_df.reset_index(drop=True)

timestamps_a = pd.to_datetime(
    aligned_a["Timestamp"].str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="raise",
)

timestamps_b = pd.to_datetime(
    aligned_b["Timestamp"].str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="raise",
)

timestamp_matches = timestamps_a.eq(timestamps_b)

feature_columns = [
    column
    for column in aligned_a.columns
    if column != "Timestamp"
]

values_a = aligned_a[feature_columns]
values_b = aligned_b[feature_columns]

feature_matches = (
    values_a.eq(values_b) | (values_a.isna() & values_b.isna())
).all(axis=1)

full_matches = timestamp_matches & feature_matches

print(f"Segment A rows: {len(segment_a_df):,}")
print(f"Segment B rows: {len(segment_b_df):,}")
print(f"Offset: {OFFSET:,} rows")
print(f"Aligned rows: {len(aligned_a):,}")
print(f"Segment A aligned start: {timestamps_a.iloc[0]}")
print(f"Segment B aligned start: {timestamps_b.iloc[0]}")
print(f"Segment A aligned end: {timestamps_a.iloc[-1]}")
print(f"Segment B aligned end: {timestamps_b.iloc[-1]}")
print(f"Matching timestamps: {int(timestamp_matches.sum()):,}")
print(f"Matching feature rows: {int(feature_matches.sum()):,}")
print(f"Matching timestamps and features: {int(full_matches.sum()):,}")
print(f"Timestamp mismatches: {int((~timestamp_matches).sum()):,}")
print(f"Feature mismatches: {int((~feature_matches).sum()):,}")

if not full_matches.all():
    mismatch_positions = np.flatnonzero(~full_matches.to_numpy())

    print("\nFirst 10 mismatches:")

    for position in mismatch_positions[:10]:
        print({
            "aligned_position": int(position),
            "segment_a_data_row": SEGMENT_A_START + OFFSET + int(position),
            "segment_b_data_row": SEGMENT_B_START + int(position),
            "timestamp_a": timestamps_a.iloc[position],
            "timestamp_b": timestamps_b.iloc[position],
            "timestamp_match": bool(timestamp_matches.iloc[position]),
            "feature_match": bool(feature_matches.iloc[position]),
        })

Segment A rows: 496,800
Segment B rows: 495,000
Offset: 1,800 rows
Aligned rows: 495,000
Segment A aligned start: 2015-12-22 16:30:00
Segment B aligned start: 2015-12-22 16:30:00
Segment A aligned end: 2015-12-28 09:59:59
Segment B aligned end: 2015-12-28 09:59:59
Matching timestamps: 495,000
Matching feature rows: 495,000
Matching timestamps and features: 495,000
Timestamp mismatches: 0
Feature mismatches: 0


### Duplicate Segment Investigation

A row-by-row comparison was conducted between the two segments of `normal.csv` containing missing values in seven variables.

The first segment contains 496,800 rows, beginning at `2015-12-22 16:00:00`. The second contains 495,000 rows, beginning at `2015-12-22 16:30:00`. Both end at `2015-12-28 09:59:59`.

After aligning the first segment by the 1,800-row difference, all 495,000 corresponding rows matched exactly across timestamps and all 45 non-missing feature columns. No timestamp or feature mismatches were found.

This establishes that the two segments are exact duplicates over their shared time interval. The additional 1,800 rows in the first segment cover the initial 30 minutes. The comparison does not establish which segment, if either, was copied from another source.

These duplicate observations have implications for downstream data preparation. Sorting or splitting the data before investigating and resolving the duplication could interleave repeated observations and compromise chronological evaluation. The appropriate treatment must be established before proceeding.

The cause of the duplication has not been established. The original CSV remains unchanged, and no rows have been removed or imputed.

**Status: Exact duplication across the shared interval verified. Cause and appropriate treatment remain under investigation.**

In [23]:
attack = pd.read_csv(
    FILES["attack"],
    encoding="utf-8-sig",
)

attack.columns = attack.columns.str.strip()

if "Timestamp" not in attack.columns:
    raise ValueError("Timestamp column not found in attack.csv.")

attack["Timestamp"] = pd.to_datetime(
    attack["Timestamp"].str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="raise",
)

duplicate_timestamps = attack["Timestamp"].duplicated(keep=False)
duplicate_rows = attack.duplicated(keep=False)

print(f"Total rows: {len(attack):,}")
print(f"Unique timestamps: {attack['Timestamp'].nunique():,}")
print(f"Rows with repeated timestamps: {int(duplicate_timestamps.sum()):,}")
print(
    "Distinct repeated timestamp values: "
    f"{attack.loc[duplicate_timestamps, 'Timestamp'].nunique():,}"
)
print(f"Rows participating in exact duplicate records: {int(duplicate_rows.sum()):,}")
print(f"Exact duplicate records beyond first occurrences: {int(attack.duplicated().sum()):,}")

chronologically_ordered = attack["Timestamp"].is_monotonic_increasing
print(f"Chronologically ordered: {chronologically_ordered}")

if not chronologically_ordered:
    backward_positions = np.flatnonzero(
        attack["Timestamp"].diff().lt(pd.Timedelta(0)).to_numpy()
    )

    print("\nFirst 20 backward timestamp transitions:")

    for position in backward_positions[:20]:
        print({
            "previous_data_row": int(position),
            "next_data_row": int(position) + 1,
            "previous_timestamp": attack["Timestamp"].iloc[position - 1],
            "next_timestamp": attack["Timestamp"].iloc[position],
        })


Total rows: 54,621
Unique timestamps: 54,621
Rows with repeated timestamps: 0
Distinct repeated timestamp values: 0
Rows participating in exact duplicate records: 0
Exact duplicate records beyond first occurrences: 0
Chronologically ordered: True


## Attack Dataset Quality Assessment

The assessment of `attack.csv` identified 54,621 records and 54,621 unique timestamps.

No repeated timestamps or exact duplicate records were found. The timestamp sequence is monotonically increasing, confirming that the file is chronologically ordered.

These results distinguish `attack.csv` from `normal.csv`, where a continuous block of missing values and duplicated observations was identified.

**Status: PASS for timestamp uniqueness, exact-record duplication, and chronological ordering.**

In [24]:
merged = pd.read_csv(
    FILES["merged"],
    encoding="utf-8-sig",
)

merged.columns = merged.columns.str.strip()

if "Timestamp" not in merged.columns:
    raise ValueError("Timestamp column not found in merged.csv.")

merged["Timestamp"] = pd.to_datetime(
    merged["Timestamp"].str.strip(),
    format="%d/%m/%Y %I:%M:%S %p",
    errors="raise",
)

duplicate_timestamps = merged["Timestamp"].duplicated(keep=False)
duplicate_rows = merged.duplicated(keep=False)

print(f"Total rows: {len(merged):,}")
print(f"Expected rows from source files: {len(attack) + missing_results['normal']['rows']:,}")
print(f"Unique timestamps: {merged['Timestamp'].nunique():,}")
print(f"Rows with repeated timestamps: {int(duplicate_timestamps.sum()):,}")
print(f"Distinct repeated timestamp values: {merged.loc[duplicate_timestamps, 'Timestamp'].nunique():,}")
print(f"Rows participating in exact duplicate records: {int(duplicate_rows.sum()):,}")
print(f"Exact duplicate records beyond first occurrences: {int(merged.duplicated().sum()):,}")
print(f"Chronologically ordered: {merged['Timestamp'].is_monotonic_increasing}")

Total rows: 1,441,719
Expected rows from source files: 1,441,719
Unique timestamps: 946,719
Rows with repeated timestamps: 990,000
Distinct repeated timestamp values: 495,000
Rows participating in exact duplicate records: 990,000
Exact duplicate records beyond first occurrences: 495,000
Chronologically ordered: False


In [25]:
def read_timestamps(path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        headers = next(csv.reader(file))

    timestamp_column = next(
        (header for header in headers if header.strip() == "Timestamp"),
        None,
    )

    if timestamp_column is None:
        raise ValueError(f"Timestamp column not found in {path.name}.")

    timestamps = pd.read_csv(
        path,
        usecols=[timestamp_column],
        encoding="utf-8-sig",
    )

    timestamps.columns = timestamps.columns.str.strip()

    return pd.to_datetime(
        timestamps["Timestamp"].str.strip(),
        format="%d/%m/%Y %I:%M:%S %p",
        errors="raise",
    ).reset_index(drop=True)


normal_timestamps = read_timestamps(FILES["normal"])
attack_timestamps = read_timestamps(FILES["attack"])
merged_timestamps = read_timestamps(FILES["merged"])

expected_timestamps = pd.concat(
    [normal_timestamps, attack_timestamps],
    ignore_index=True,
)

print("Row counts:")
print(f"normal.csv:  {len(normal_timestamps):,}")
print(f"attack.csv:  {len(attack_timestamps):,}")
print(f"merged.csv:  {len(merged_timestamps):,}")
print(f"Expected concatenation: {len(expected_timestamps):,}")

if len(merged_timestamps) != len(expected_timestamps):
    raise ValueError("Merged row count differs from the expected concatenation.")

print(
    "\nFull timestamp sequence matches normal + attack:",
    merged_timestamps.equals(expected_timestamps),
)

backward_positions = merged_timestamps.diff().lt(pd.Timedelta(0))
backward_indices = backward_positions[backward_positions].index

print(f"\nBackward timestamp transitions: {len(backward_indices):,}")

for position in backward_indices[:20]:
    print({
        "previous_data_row": int(position),
        "next_data_row": int(position) + 1,
        "previous_timestamp": merged_timestamps.iloc[position - 1],
        "next_timestamp": merged_timestamps.iloc[position],
    })

boundary_positions = [0, 395_298, 892_098, 1_387_097, 1_387_098, 1_441_718]

print("\nBoundary spot-check (1-indexed data rows):")

for position in boundary_positions:
    if position < len(merged_timestamps):
        print({
            "data_row": position + 1,
            "merged_timestamp": merged_timestamps.iloc[position],
            "expected_timestamp": expected_timestamps.iloc[position],
            "match": merged_timestamps.iloc[position] == expected_timestamps.iloc[position],
        })

Row counts:
normal.csv:  1,387,098
attack.csv:  54,621
merged.csv:  1,441,719
Expected concatenation: 1,441,719

Full timestamp sequence matches normal + attack: True

Backward timestamp transitions: 2
{'previous_data_row': 395298, 'next_data_row': 395299, 'previous_timestamp': Timestamp('2016-01-02 14:59:59'), 'next_timestamp': Timestamp('2015-12-22 16:00:00')}
{'previous_data_row': 892098, 'next_data_row': 892099, 'previous_timestamp': Timestamp('2015-12-28 09:59:59'), 'next_timestamp': Timestamp('2015-12-22 16:30:00')}

Boundary spot-check (1-indexed data rows):
{'data_row': 1, 'merged_timestamp': Timestamp('2015-12-28 10:00:00'), 'expected_timestamp': Timestamp('2015-12-28 10:00:00'), 'match': True}
{'data_row': 395299, 'merged_timestamp': Timestamp('2015-12-22 16:00:00'), 'expected_timestamp': Timestamp('2015-12-22 16:00:00'), 'match': True}
{'data_row': 892099, 'merged_timestamp': Timestamp('2015-12-22 16:30:00'), 'expected_timestamp': Timestamp('2015-12-22 16:30:00'), 'match': T

## Merged Dataset Timestamp Reconciliation

The timestamp sequence in `merged.csv` was compared with the concatenated timestamp sequences of `normal.csv` and `attack.csv`.

The three files contain 1,387,098, 54,621, and 1,441,719 records, respectively. The row count of `merged.csv` matches the combined source-file row count.

The complete timestamp sequence in `merged.csv` matches `normal.csv` followed by `attack.csv`. All six selected boundary checks also matched the expected timestamps.

Two backward timestamp transitions remain in the merged sequence. Both occur within the `normal.csv` portion and correspond to the previously verified segment boundaries. The merged sequence then continues with the records from `attack.csv`.

These results establish the positional concatenation order of the timestamp sequences. They do not yet establish that all feature values in `merged.csv` match their corresponding source records.

**Status: Timestamp-sequence reconciliation passed; full feature-level reconciliation remains outstanding.**

In [ ]:
from itertools import chain

def read_chunks(path):
    for chunk in pd.read_csv(
        path,
        chunksize=CHUNK_SIZE,
        encoding="utf-8-sig",
    ):
        chunk.columns = chunk.columns.str.strip()
        yield chunk

source_chunks = chain(
    read_chunks(FILES["normal"]),
    read_chunks(FILES["attack"]),
)

source_buffer = next(source_chunks, None)
source_offset = 0

total_rows = 0
mismatched_rows = 0
mismatched_cells = 0
first_mismatches = []

for merged_chunk in read_chunks(FILES["merged"]):
    merged_chunk = merged_chunk.reset_index(drop=True)

    if source_buffer is None:
        raise ValueError("merged.csv contains more rows than the source files.")

    expected_parts = []
    rows_needed = len(merged_chunk)

    while rows_needed > 0:
        if source_buffer is None:
            raise ValueError("The source files contain fewer rows than merged.csv.")

        available = len(source_buffer) - source_offset
        rows_to_take = min(rows_needed, available)

        expected_parts.append(
            source_buffer.iloc[
                source_offset : source_offset + rows_to_take
            ]
        )

        source_offset += rows_to_take
        rows_needed -= rows_to_take

        if source_offset == len(source_buffer):
            source_buffer = next(source_chunks, None)
            source_offset = 0

    expected_chunk = pd.concat(expected_parts, ignore_index=True)

    if list(expected_chunk.columns) != list(merged_chunk.columns):
        raise ValueError("Column names or column order do not match.")

    expected_normalized = expected_chunk.copy()
    merged_normalized = merged_chunk.copy()

    for column in expected_normalized.columns:
        if column == "Timestamp":
            expected_normalized[column] = pd.to_datetime(
                expected_normalized[column].str.strip(),
                format="%d/%m/%Y %I:%M:%S %p",
                errors="raise",
            )
            merged_normalized[column] = pd.to_datetime(
                merged_normalized[column].str.strip(),
                format="%d/%m/%Y %I:%M:%S %p",
                errors="raise",
            )
        else:
            expected_values = pd.to_numeric(
                expected_normalized[column],
                errors="coerce",
            )
            merged_values = pd.to_numeric(
                merged_normalized[column],
                errors="coerce",
            )

            both_numeric = (
                expected_values.notna() | merged_values.notna()
            )

            expected_normalized[column] = expected_values.where(
                both_numeric,
                expected_normalized[column].str.strip(),
            )
            merged_normalized[column] = merged_values.where(
                both_numeric,
                merged_normalized[column].str.strip(),
            )

    cell_matches = (
        expected_normalized.eq(merged_normalized)
        | (
            expected_normalized.isna()
            & merged_normalized.isna()
        )
    )

    expected_numeric["Timestamp"] = pd.to_datetime(
        expected_numeric["Timestamp"].str.strip(),
        format="%d/%m/%Y %I:%M:%S %p",
        errors="raise",
    )

    merged_numeric["Timestamp"] = pd.to_datetime(
        merged_numeric["Timestamp"].str.strip(),
        format="%d/%m/%Y %I:%M:%S %p",
        errors="raise",
    )

    numeric_columns = [
        column for column in expected_numeric.columns
        if column != "Timestamp"
    ]

    expected_numeric[numeric_columns] = expected_numeric[numeric_columns].apply(
        pd.to_numeric, errors="raise"
    )

    merged_numeric[numeric_columns] = merged_numeric[numeric_columns].apply(
        pd.to_numeric, errors="raise"
    )

    cell_matches = (
        expected_numeric.eq(merged_numeric)
        | (
            expected_numeric.isna()
            & merged_numeric.isna()
        )
    )

    row_matches = cell_matches.all(axis=1)

    mismatched_cells += int((~cell_matches).to_numpy().sum())
    mismatched_rows += int((~row_matches).sum())

    if len(first_mismatches) < 10:
        for position in row_matches.index[~row_matches]:
            differing_columns = cell_matches.columns[
                ~cell_matches.loc[position]
            ].tolist()

            first_mismatches.append(
                {
                    "data_row": total_rows + int(position) + 1,
                    "columns": differing_columns,
                }
            )

            if len(first_mismatches) == 10:
                break

    total_rows += len(merged_chunk)

if source_buffer is not None:
    raise ValueError("The source files contain more rows than merged.csv.")

print(f"Rows compared: {total_rows:,}")
print(f"Columns compared: {len(merged_chunk.columns)}")
print(f"Rows with mismatches: {mismatched_rows:,}")
print(f"Mismatched cells: {mismatched_cells:,}")
print(f"First mismatches: {first_mismatches}")

if mismatched_rows == 0:
    print("RESULT: Full numeric reconciliation PASSED.")
else:
    print("RESULT: Full numeric reconciliation FAILED.")

ValueError: Unable to parse string "Normal" at position 0

In [28]:
CHECK_COLUMNS = [
    "MV101", "AIT201", "MV201", "P201",
    "P202", "P204", "MV303", "FIT101",
]

normal_sample = pd.read_csv(
    FILES["normal"],
    nrows=5,
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)
normal_sample.columns = normal_sample.columns.str.strip()

merged_sample = pd.read_csv(
    FILES["merged"],
    nrows=5,
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)
merged_sample.columns = merged_sample.columns.str.strip()

print("NORMAL.CSV — first five rows")
print(normal_sample[CHECK_COLUMNS].to_string(index=False))

print("\nMERGED.CSV — first five rows")
print(merged_sample[CHECK_COLUMNS].to_string(index=False))

NORMAL.CSV — first five rows
MV101   AIT201 MV201 P201 P202 P204 MV303   FIT101
    2 262.0161     2    1    1    1     1 2.427057
    2 262.0161     2    1    1    1     1 2.446274
    2 262.0161     2    1    1    1     1 2.489191
    2 262.0161     2    1    1    1     1  2.53435
    2 262.0161     2    1    1    1     1  2.56926

MERGED.CSV — first five rows
MV101   AIT201 MV201 P201 P202 P204 MV303   FIT101
  2.0 262.0161   2.0  1.0  1.0  1.0   1.0 2.427057
  2.0 262.0161   2.0  1.0  1.0  1.0   1.0 2.446274
  2.0 262.0161   2.0  1.0  1.0  1.0   1.0 2.489191
  2.0 262.0161   2.0  1.0  1.0  1.0   1.0  2.53435
  2.0 262.0161   2.0  1.0  1.0  1.0   1.0  2.56926


In [29]:
CHECK_COLUMNS = [
    "MV101", "AIT201", "MV201", "P201",
    "P202", "P204", "MV303", "FIT101",
]

normal_sample = pd.read_csv(
    FILES["normal"],
    nrows=5,
    encoding="utf-8-sig",
)
normal_sample.columns = normal_sample.columns.str.strip()

merged_sample = pd.read_csv(
    FILES["merged"],
    nrows=5,
    encoding="utf-8-sig",
)
merged_sample.columns = merged_sample.columns.str.strip()

normal_values = normal_sample[CHECK_COLUMNS].apply(
    pd.to_numeric, errors="raise"
)
merged_values = merged_sample[CHECK_COLUMNS].apply(
    pd.to_numeric, errors="raise"
)

print("Numeric values equal:", normal_values.equals(merged_values))
print(
    "Maximum absolute difference:",
    (normal_values - merged_values).abs().to_numpy().max(),
)

Numeric values equal: False
Maximum absolute difference: 0.0


In [30]:
print("NORMAL data types:")
print(normal_values.dtypes)

print("\nMERGED data types:")
print(merged_values.dtypes)

print("\nElement-by-element numeric equality:")
print(normal_values.eq(merged_values).all().all())

NORMAL data types:
MV101       int64
AIT201    float64
MV201       int64
P201        int64
P202        int64
P204        int64
MV303       int64
FIT101    float64
dtype: object

MERGED data types:
MV101     float64
AIT201    float64
MV201     float64
P201      float64
P202      float64
P204      float64
MV303     float64
FIT101    float64
dtype: object

Element-by-element numeric equality:
True
